In [1]:
# STEP 2: Fetch iFixit page and preview text
import requests
from bs4 import BeautifulSoup

url = "https://www.ifixit.com/Teardown/iPhone+12+and+12+Pro+Teardown/137669"
headers = {"User-Agent": "SmartBinXBot/1.0 (+student project)"}

response = requests.get(url, headers=headers)
print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "lxml")
paragraphs = [p.get_text(" ", strip=True) for p in soup.select("p")]

print("\n--- First 5 paragraphs ---\n")
for p in paragraphs[:5]:
    print(p)

Status code: 200

--- First 5 paragraphs ---

Teardown
Thanks for joining us for a live teardown of the iPhone 12 and 12 Pro! If you missed the livestream, no worries—you can still catch the recording above. Or scroll down for the written analysis, including some bonus disassembly of the new MagSafe power puck. And if that doesn’t quite quench your iPhone 12 teardown thirst, check out our teardowns of the iPhone 12 mini and the iPhone 12 Pro Max !
Be sure to follow iFixit’s YouTube channel , our Instagram , and our Twitter , and subscribe to our newsletter so you’ll be the first to know when the newest consumer tech hits the teardown table.
This teardown is not a repair guide. To repair your iPhone 12 Pro, use our service manual .
While we wait with bated breath for the Mini and Max to show up, we can at least get started on the inbetweeners. Let's see what they're packin':


In [2]:
# STEP 3: Search for material words
text = " ".join(paragraphs).lower()
materials = ["aluminum","copper","gold","silver","plastic","glass","lithium","steel","iron","magnesium","rare earth"]
found = [m for m in materials if m in text]
print("Materials found in this page:", found)

Materials found in this page: ['aluminum', 'copper', 'plastic', 'glass', 'steel']


In [3]:
# Convert found keywords into heuristic percentages, save into DB, and show saved row
import re, json, sqlite3
from datetime import datetime

DB_PATH = "smartbinx_full.db"
model_display = "iPhone 12 and 12 Pro"
model_norm = re.sub(r'[^a-z0-9 ]', '', model_display.lower()).strip()
url = "https://www.ifixit.com/Teardown/iPhone+12+and+12+Pro+Teardown/137669"

# The materials your previous cell found:
found_keywords = ['aluminum', 'copper', 'plastic', 'glass', 'steel']

# Heuristic template & adjustment
default = {"Plastic":30, "Glass":15, "Aluminum":10, "Copper":12, "Gold":0.02, "Silver":0.04, "Lithium":8, "Steel/Iron":5, "Others":19}
res = dict(default)

# bump values for detected keywords
fk = {k.lower() for k in found_keywords}
if 'aluminum' in fk or 'aluminium' in fk:
    res['Aluminum'] = min(res.get('Aluminum',5)+8, 60)
if 'copper' in fk:
    res['Copper'] = min(res.get('Copper',10)+8, 60)
if 'gold' in fk:
    res['Gold'] = max(res.get('Gold',0.01), 0.02)
if 'silver' in fk:
    res['Silver'] = max(res.get('Silver',0.01), 0.02)
if 'plastic' in fk:
    res['Plastic'] = min(res.get('Plastic',20)+10, 90)
if 'glass' in fk:
    res['Glass'] = min(res.get('Glass',5)+15, 90)
if 'lithium' in fk or 'battery' in fk:
    res['Lithium'] = min(res.get('Lithium',4)+6, 40)
if 'steel' in fk or 'iron' in fk:
    res['Steel/Iron'] = min(res.get('Steel/Iron',5)+8, 60)

# Normalize to 100%
total = sum(v for v in res.values() if isinstance(v, (int,float)))
materials_pct = {k: round((v/total)*100, 2) for k,v in res.items()}

print("Estimated material percentages (heuristic):")
for k,v in materials_pct.items():
    print(f" - {k}: {v}%")

# Save into DB
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS ewaste_models (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    model_name_normalized TEXT UNIQUE,
    display_name TEXT,
    source TEXT,
    source_url TEXT,
    materials_json TEXT,
    notes TEXT,
    confidence REAL,
    last_updated TEXT
)
""")
now = datetime.now().isoformat()
mj = json.dumps(materials_pct, ensure_ascii=False)
cur.execute("SELECT id FROM ewaste_models WHERE model_name_normalized = ?", (model_norm,))
row = cur.fetchone()
if row:
    cur.execute("""UPDATE ewaste_models SET display_name=?, source=?, source_url=?, materials_json=?, notes=?, confidence=?, last_updated=? WHERE id=?""",
                (model_display, "ifixit (parsed)", url, mj, "Automated parse from iFixit page", 0.6, now, row[0]))
    model_id = row[0]
else:
    cur.execute("""INSERT INTO ewaste_models (model_name_normalized, display_name, source, source_url, materials_json, notes, confidence, last_updated)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                (model_norm, model_display, "ifixit (parsed)", url, mj, "Automated parse from iFixit page", 0.6, now))
    model_id = cur.lastrowid
conn.commit()

# show saved row
cur.execute("SELECT id, display_name, source, materials_json, last_updated FROM ewaste_models WHERE id = ?", (model_id,))
r = cur.fetchone()
conn.close()
print("\nSaved to DB (id):", r[0])
print("Display name:", r[1])
print("Source:", r[2])
print("Materials (saved):", json.loads(r[3]))
print("Last updated:", r[4])

Estimated material percentages (heuristic):
 - Plastic: 27.02%
 - Glass: 20.26%
 - Aluminum: 12.16%
 - Copper: 13.51%
 - Gold: 0.01%
 - Silver: 0.03%
 - Lithium: 5.4%
 - Steel/Iron: 8.78%
 - Others: 12.83%

Saved to DB (id): 2
Display name: iPhone 12 and 12 Pro
Source: ifixit (parsed)
Materials (saved): {'Plastic': 27.02, 'Glass': 20.26, 'Aluminum': 12.16, 'Copper': 13.51, 'Gold': 0.01, 'Silver': 0.03, 'Lithium': 5.4, 'Steel/Iron': 8.78, 'Others': 12.83}
Last updated: 2025-10-24T16:30:38.719320
